# dataset

In [2]:
import numpy as np
import h5py

def get_global_min_max(file_paths, batch_size=2000):
    """
    Iterates through files to find global min and max for DL and UL
    by explicitly accessing the 'real' and 'imag' fields of the structured array.
    """
    # Initialize with extreme values
    global_stats = {
        'dl': {'min': float('inf'), 'max': float('-inf')},
        'ul': {'min': float('inf'), 'max': float('-inf')}
    }
    
    for path in file_paths:
        print(f"Scanning {path} for min/max stats...")
        with h5py.File(path, 'r') as f:
            for key in ['csi_dl', 'csi_ul']:
                dataset = f[key] 
                num_samples = dataset.shape[3]
                stat_key = 'dl' if key == 'csi_dl' else 'ul'
                
                # Iterate in chunks to save RAM
                for i in range(0, num_samples, batch_size):
                    end = min(i + batch_size, num_samples)
                    
                    # 1. Load the structured chunk
                    # Shape: (2, 256, 32, batch_size) with dtype=[('real', '<f4'), ('imag', '<f4')]
                    chunk_struct = dataset[:, :, :, i:end]
                    
                    # 2. Extract numerical components
                    # We want the global min/max across ALL values (real and imaginary)
                    real_part = chunk_struct['real']
                    imag_part = chunk_struct['imag']
                    
                    # 3. Compute min/max for this chunk
                    # Check both real and imag parts to find the absolute extremes
                    chunk_min = min(np.min(real_part), np.min(imag_part))
                    chunk_max = max(np.max(real_part), np.max(imag_part))
                    
                    # 4. Update global stats
                    if chunk_min < global_stats[stat_key]['min']:
                        global_stats[stat_key]['min'] = float(chunk_min)
                    if chunk_max > global_stats[stat_key]['max']:
                        global_stats[stat_key]['max'] = float(chunk_max)
                        
    print("\nGlobal Statistics Calculated:")
    print(global_stats)
    return global_stats

# Define your file paths
train_path = "../train_data-002.mat"
val_path = "../val_data-003.mat"
test_path = "../test_data-001.mat"

# Execute calculation
stats = get_global_min_max([train_path, val_path, test_path])

Scanning ../train_data-002.mat for min/max stats...
Scanning ../val_data-003.mat for min/max stats...
Scanning ../test_data-001.mat for min/max stats...

Global Statistics Calculated:
{'dl': {'min': -0.08417291194200516, 'max': 0.08958114683628082}, 'ul': {'min': -0.08988773077726364, 'max': 0.08599823713302612}}


In [3]:
import torch

class CSIDataCollaterLazyNormalized:
    def __init__(self, train_path, val_path, test_path, stats, batch_size=200, snr_low=-10.0, snr_high=10.0):
        self.batch_size = batch_size
        self.snr_low = snr_low
        self.snr_high = snr_high
        self.stats = stats  # Dictionary containing global min/max
        
        self.paths = {
            'train': train_path,
            'val': val_path,
            'test': test_path
        }
        
        self.files = {}
        self.datasets = {}
        self.lengths = {}
        
        for mode, path in self.paths.items():
            f = h5py.File(path, 'r')
            self.files[mode] = f
            
            if 'csi_dl' not in f.keys() or 'csi_ul' not in f.keys():
                raise KeyError(f"{path} missing 'csi_dl' or 'csi_ul'")
            
            self.datasets[mode] = {
                'dl': f['csi_dl'],
                'ul': f['csi_ul']
            }
            self.lengths[mode] = f['csi_dl'].shape[3]
            print(f"Initialized {mode} normalized loader: {self.lengths[mode]} samples.")

    def _normalize(self, data, min_val, max_val):
        """
        Applies Min-Max normalization to range [-1, 1].
        Formula: 2 * (x - min) / (max - min) - 1
        """
        # Avoid division by zero if max == min (unlikely in real data)
        denominator = max_val - min_val if max_val != min_val else 1.0
        
        norm_data = (data - min_val) / denominator
        norm_data = norm_data * 2 - 1
        return norm_data

    def _process_batch_data(self, batch_arr, mode_key):
        """
        Processes a raw batch:
        1. Converts complex struct to float32
        2. Stacks Real/Imag
        3. Transposes to (B, C, H, W)
        4. Normalizes to [-1, 1] using global stats
        """
        # 1. Extract Real/Imag
        arr_real = batch_arr['real'].astype(np.float32)
        arr_imag = batch_arr['imag'].astype(np.float32)
        
        # 2. Stack: (B, 2, ...)
        # Note: Your raw shape was (2, 256, 32, B). 
        # Stack logic depends on how numpy loads the struct. 
        # Assuming arr_real is (2, 256, 32, B) based on previous code.
        
        batch_arr_stacked = np.stack([arr_real, arr_imag], axis=-1) # Check axis carefully based on input
        # Actually, in your previous code: batch_arr was struct array.
        # arr_real shape is (2, 256, 32, B).
        # We want final shape (B, 2, 32, 256) or similar.
        
        # Replicating your previous logic exactly but adding normalization:
        # Previous logic: np.stack([arr_real, arr_imag], axis=2) -> squeeze -> transpose
        
        # Let's clean the shape transformation:
        # Input: (2, 256, 32, B)
        # Target: (B, 13, 32, 2) -> Transposed to (B, 2, 32, 256)
        
        # Combining Real/Imag parts usually implies we treat them as channels or complex.
        # Let's perform normalization on the raw float values first
        min_v = self.stats[mode_key]['min']
        max_v = self.stats[mode_key]['max']
        
        arr_real = self._normalize(arr_real, min_v, max_v)
        arr_imag = self._normalize(arr_imag, min_v, max_v)

        # Reconstruct Batch
        # Stack Real and Imag into a new axis. 
        # Current shape: (2, 256, 32, B)
        # We want to eventually get to (B, 2, 32, 256) as per your ATN input.
        
        # Combine back to complex-like structure for transpose steps
        # If your previous code worked, let's stick to that flow:
        batch_arr_comb = np.stack([arr_real, arr_imag], axis=2) 
        batch_arr_comb = np.squeeze(batch_arr_comb) # Handling the specific squeezing
        
        # Transpose to (B, 2, 32, 256)
        # Assuming original shape flow: (2, 256, 32, B) -> Transpose to (B, ...)
        # The safest transpose for (2, 256, 32, B) to (B, 2, 32, 256):
        batch_arr_comb = np.transpose(batch_arr_comb, (3, 2, 0, 1)) 
        
        return batch_arr_comb

    def __call__(self, mode="train"):
        ds_dl = self.datasets[mode]['dl']
        ds_ul = self.datasets[mode]['ul']
        N = self.lengths[mode]
        
        raw_indices = np.random.choice(N, self.batch_size, replace=False)
        raw_indices.sort() 
        
        # Read from Disk
        batch_dl_raw = ds_dl[:,:,:,raw_indices]
        batch_ul_raw = ds_ul[:,:,:,raw_indices]
        
        # Process and Normalize
        # We pass 'dl' or 'ul' key to use correct min/max stats
        batch_dl = self._process_batch_data(batch_dl_raw, 'dl')
        batch_ul = self._process_batch_data(batch_ul_raw, 'ul')
        
        # Generate SNR
        batch_snr = np.random.uniform(self.snr_low, self.snr_high, (self.batch_size, 1))
        
        return (torch.from_numpy(batch_dl).float(), 
                torch.from_numpy(batch_ul).float(), 
                torch.from_numpy(batch_snr).float())

    def close(self):
        for f in self.files.values():
            f.close()

In [4]:
# Usage Example:
train_file = "../train_data-002.mat"
val_file = "../val_data-003.mat"
test_file = "../test_data-001.mat"

collater = CSIDataCollaterLazyNormalized(
    train_file, val_file, test_file, 
    stats=stats,  # Pass the calculated stats here
    batch_size=200
)

Initialized train normalized loader: 80000 samples.
Initialized val normalized loader: 30000 samples.
Initialized test normalized loader: 20000 samples.


In [5]:
dl_batch, ul_batch, snr_batch = collater(mode="train")

print("Min value in batch:", dl_batch.min().item())
print("Max value in batch:", dl_batch.max().item())

Min value in batch: -0.13112002611160278
Max value in batch: 0.08014273643493652


In [6]:
dl_batch.shape, ul_batch.shape, snr_batch.shape

(torch.Size([200, 2, 256, 32]),
 torch.Size([200, 2, 256, 32]),
 torch.Size([200, 1]))

## AF module

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AFModule(nn.Module):
    def __init__(self, channels, reduction_ratio=2):
        """
        AF Module (Attention Feature Module)
        Args:
            channels: Number of input feature channels (C).
            reduction_ratio: Ratio for the hidden layer size in the MLP. 
                             Since channels are small (16), we can use a small ratio or no reduction.
        """
        super(AFModule, self).__init__()
        
        # 1. Context Extraction is handled in forward (Global Avg Pooling + Concat)
        
        # 2. Factor Prediction (MLP)
        # Input size = Channels (from Global Avg Pool) + 1 (from SNR mu)
        input_dim = channels + 1 
        hidden_dim = max(channels // reduction_ratio, 1) # Ensure hidden dim is at least 1
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, channels)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, snr):
        """
        Args:
            x: Feature map tensor of shape (Batch, Channels, Height, Width)
            snr: Signal-to-Noise Ratio tensor of shape (Batch, 1)
        """
        batch, channels, height, width = x.size()

        # --- 1. Context Extraction ---
        # Global Average Pooling: Average over H and W dimensions -> (Batch, Channels)
        global_feat = F.avg_pool2d(x, (height, width)).view(batch, channels)
        
        # Concatenation: Combine global features with SNR
        # global_feat: (B, C), snr: (B, 1) -> context: (B, C+1)
        context = torch.cat([global_feat, snr], dim=1)

        # --- 2. Factor Prediction ---
        # Pass through MLP to get scale factors
        out = self.fc1(context)
        out = self.relu(out)
        out = self.fc2(out)
        scale_factors = self.sigmoid(out) # Shape: (Batch, Channels)

        # --- 3. Feature Recalibration ---
        # Reshape scale factors to (Batch, Channels, 1, 1) for broadcasting
        scale_factors = scale_factors.view(batch, channels, 1, 1)
        
        # Element-wise multiplication
        out = x * scale_factors
        
        return out



## ATN module

In [8]:
class ATN(nn.Module):
    def __init__(self):
        super(ATN, self).__init__()
        
        # --- Block 1 ---
        # Conv: 2 -> 16, Kernel (3,3), Stride (2,1), Padding (1,1)
        self.conv1 = nn.Conv2d(in_channels=2, out_channels=16, 
                               kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU()
        self.af1 = AFModule(channels=16)

        # --- Block 2 ---
        # Conv: 16 -> 16, Kernel (3,3), Stride (2,1), Padding (1,1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=16, 
                               kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.bn2 = nn.BatchNorm2d(16)
        self.prelu2 = nn.PReLU()
        self.af2 = AFModule(channels=16)

        # --- Block 3 ---
        # Conv: 16 -> 2, Kernel (3,3), Stride (2,1), Padding (1,1)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=2, 
                               kernel_size=(3, 3), stride=(2, 1), padding=(1, 1))
        self.bn3 = nn.BatchNorm2d(2)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, snr):
        """
        Args:
            x: Input tensor of shape (Batch, Height, Width, Channels) -> (B, 13, 32, 2)
            snr: SNR value (Batch, 1)
        """
        
        # --- Block 1 ---
        x = self.conv1(x)       # (B, 2, 13, 32) -> (B, 16, 7, 32)
        x = self.bn1(x)
        x = self.prelu1(x)
        x = self.af1(x, snr)    # AF Module uses 'snr'

        # --- Block 2 ---
        x = self.conv2(x)       # (B, 16, 7, 32) -> (B, 16, 4, 32)
        x = self.bn2(x)
        x = self.prelu2(x)
        x = self.af2(x, snr)    # AF Module uses 'snr'

        # --- Block 3 ---
        x = self.conv3(x)       # (B, 16, 4, 32) -> (B, 2, 2, 32)
        x = self.bn3(x)
        x = self.sigmoid(x)

        
        return x



In [9]:

if __name__ == "__main__":
    # 1. Setup Input with Batch Size = 200
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size = 200
    
    # Input format: (Batch, Height, Width, Channels) -> (200, 13, 32, 2)
    input_tensor = dl_batch.to(device)
    
    # 2. Setup SNR (mu)
    # One SNR value per image in the batch -> (200, 1)
    snr_tensor = snr_batch.to(device)

    # 3. Initialize Model
    model = ATN().to(device)

    # 4. Forward Pass
    output_tensor = model(input_tensor, snr_tensor)

    # 5. Print Dimensions
    print("--- Dimension Check ---")
    print(f"Input Shape:  {input_tensor.shape}")  # Expected: (200, 13, 32, 2)
    print(f"Output Shape: {output_tensor.shape}") # Expected: (200, 2, 32, 2)


--- Dimension Check ---
Input Shape:  torch.Size([200, 2, 256, 32])
Output Shape: torch.Size([200, 2, 32, 32])


## encoder

In [10]:
class CsiNetPlusEncoderWithAF(nn.Module):
    def __init__(self, compression_ratio=16):
        """
        ADJSCC-CSI Encoder with Attention Feature (AF) Modules.
        Input shape: [B, 2, 32, 32]
        """
        super(CsiNetPlusEncoderWithAF, self).__init__()
        
        # --- Constants ---
        self.input_channels = 2
        self.height = 32
        self.width = 32
        self.total_elements = self.input_channels * self.height * self.width # 2048
        self.M = int(self.total_elements / compression_ratio)
        
        # --- 1. First Convolutional Block ---
        self.conv1 = nn.Conv2d(in_channels=self.input_channels, 
                               out_channels=2, 
                               kernel_size=7, 
                               stride=1, 
                               padding=3)
        self.bn1 = nn.BatchNorm2d(num_features=2)
        self.act1 = nn.LeakyReLU(negative_slope=0.3, inplace=True)
        
        # [NEW] AF Module inserted after activation [cite: 2737, 2803]
        self.af1 = AFModule(channels=2)
        
        # --- 2. Second Convolutional Block ---
        self.conv2 = nn.Conv2d(in_channels=2, 
                               out_channels=2, 
                               kernel_size=7, 
                               stride=1, 
                               padding=3)
        self.bn2 = nn.BatchNorm2d(num_features=2)
        self.act2 = nn.LeakyReLU(negative_slope=0.3, inplace=True)
        
        # [NEW] AF Module inserted after activation [cite: 2737, 2803]
        self.af2 = AFModule(channels=2)
        
        # --- 3. Fully Connected Layer ---
        self.fc = nn.Linear(in_features=self.total_elements, out_features=self.M)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, snr):
        """
        Forward pass with SNR adaption.
        Args:
            x: Input tensor [B, 2, 32, 32]
            snr: SNR value tensor [B, 1] required for AF Modules
        """
        
        # Step 1: First Conv Block + AF Recalibration
        out = self.conv1(x)
        out = self.bn1(out) 
        out = self.act1(out)
        out = self.af1(out, snr) # Apply Attention Feature module [cite: 2737]
        
        # Step 2: Second Conv Block + AF Recalibration
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.act2(out)
        out = self.af2(out, snr) # Apply Attention Feature module [cite: 2737]
        
        # Step 3: Flatten
        out = out.view(out.size(0), -1)
        
        # Step 4: FC Layer + Scaling
        out = self.fc(out)
        out = self.sigmoid(out)
        
        return out



In [11]:
# --- Verification Example ---
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Instantiate Modified Encoder
    model = CsiNetPlusEncoderWithAF(compression_ratio=16).to(device)
    
    # Dummy Input [B, 32, 32, 2]
    dummy_input = output_tensor.to(device)

    dummy_snr = snr_batch.to(device)
    
    # Forward Pass with SNR
    output = model(dummy_input, dummy_snr)
    
    print(f"Input Shape: {dummy_input.shape}")
    print(f"SNR Shape: {dummy_snr.shape}")
    print(f"Output Shape: {output.shape}")
    print(f"AF Modules Successfully Integrated.")

Input Shape: torch.Size([200, 2, 32, 32])
SNR Shape: torch.Size([200, 1])
Output Shape: torch.Size([200, 128])
AF Modules Successfully Integrated.


In [12]:
output[0]

tensor([0.5486, 0.5856, 0.4720, 0.6212, 0.5358, 0.5618, 0.4684, 0.5502, 0.5210,
        0.4724, 0.4288, 0.4487, 0.4450, 0.5067, 0.5325, 0.5319, 0.4509, 0.4611,
        0.5854, 0.4839, 0.4658, 0.4543, 0.5346, 0.4991, 0.4883, 0.6405, 0.4226,
        0.5117, 0.5594, 0.5241, 0.4135, 0.5131, 0.4648, 0.5843, 0.4280, 0.5544,
        0.4678, 0.5543, 0.4681, 0.6667, 0.4488, 0.4759, 0.5923, 0.6048, 0.5311,
        0.6193, 0.4710, 0.4720, 0.5369, 0.5469, 0.5851, 0.5499, 0.5641, 0.4778,
        0.4714, 0.5033, 0.5192, 0.5115, 0.5310, 0.6005, 0.5263, 0.5627, 0.5257,
        0.6306, 0.5385, 0.5534, 0.4556, 0.4825, 0.5065, 0.5482, 0.5818, 0.4757,
        0.5485, 0.4531, 0.4823, 0.5214, 0.5168, 0.4762, 0.5198, 0.5088, 0.4939,
        0.5746, 0.4513, 0.5783, 0.6169, 0.5031, 0.4619, 0.5671, 0.3990, 0.3968,
        0.4673, 0.5377, 0.4729, 0.5240, 0.4527, 0.4560, 0.5582, 0.5125, 0.5131,
        0.4639, 0.3702, 0.4692, 0.6239, 0.5473, 0.3942, 0.5855, 0.6221, 0.4511,
        0.4303, 0.5320, 0.5983, 0.6215, 

## real to complex symbols and power normalization

In [13]:
import torch

def enc_to_complex_and_normalize(encoder_output):
    """
    Converts real-valued encoder output to complex representation and 
    applies power normalization.

    Args:
        encoder_output (torch.Tensor): Output from encoder with shape [B, 2k] 
                                       (e.g., [B, 16]).
    
    Returns:
        torch.Tensor: Normalized complex symbols with shape [B, k] (e.g., [B, 8]).
    """
    
    # --- Step 1: Real-to-Complex Conversion [cite: 146, 324] ---
    # The paper mentions combining real values to form complex values.
    # As discussed, we split the [B, 16] vector into two [B, 8] vectors.
    
    # Calculate k (half the dimension of the real vector)
    k = encoder_output.shape[1] // 2 
    
    # Slicing: First k elements = Real (I), Remaining k elements = Imaginary (Q)
    real_part = encoder_output[:, :k]
    imag_part = encoder_output[:, k:]
    
    # Create the complex tensor s
    s = torch.complex(real_part, imag_part)
    
    # --- Step 2: Power Normalization  ---
    # Constraint: (1/k) * E(s * s*) = 1
    # We normalize by the square root of the average power over the batch 
    # to satisfy the expectation E[...] = 1.
    
    # Calculate the average power of the current batch
    # |s|^2 = real^2 + imag^2
    batch_avg_power = torch.mean(s.abs().square())
    
    # Normalize the complex vector s
    # We divide by sqrt(P_avg) so that the new P_avg becomes 1
    s_normalized = s / torch.sqrt(batch_avg_power)
    
    return s_normalized



In [14]:

encoder_output = output

# Apply conversion and normalization
complex_symbols = enc_to_complex_and_normalize(encoder_output)

print("Input Shape:      ", encoder_output.shape)  # Should be [4, 16]
print("Output Shape:     ", complex_symbols.shape) # Should be [4, 8]
print("Output Data Type: ", complex_symbols.dtype) # Should be complex64

# --- Verification of Power Constraint ---
# The mean power of the output should be approximately 1.0
actual_power = torch.mean(complex_symbols.abs().square())
print(f"Average Power:     {actual_power.item():.4f}")

Input Shape:       torch.Size([200, 128])
Output Shape:      torch.Size([200, 64])
Output Data Type:  torch.complex64
Average Power:     1.0000


In [15]:
complex_symbols[0]

tensor([0.7485+0.7347j, 0.7991+0.7550j, 0.6441+0.6216j, 0.8476+0.6584j,
        0.7310+0.6911j, 0.7665+0.7480j, 0.6391+0.7938j, 0.7507+0.6491j,
        0.7109+0.7484j, 0.6445+0.6182j, 0.5850+0.6581j, 0.6122+0.7113j,
        0.6072+0.7052j, 0.6913+0.6497j, 0.7266+0.7092j, 0.7257+0.6943j,
        0.6153+0.6739j, 0.6291+0.7840j, 0.7988+0.6158j, 0.6602+0.7890j,
        0.6356+0.8417j, 0.6199+0.6864j, 0.7294+0.6302j, 0.6810+0.7738j,
        0.6663+0.5444j, 0.8739+0.5414j, 0.5766+0.6376j, 0.6981+0.7336j,
        0.7633+0.6452j, 0.7151+0.7149j, 0.5642+0.6177j, 0.7000+0.6222j,
        0.6341+0.7616j, 0.7972+0.6993j, 0.5840+0.7001j, 0.7564+0.6330j,
        0.6382+0.5052j, 0.7563+0.6402j, 0.6387+0.8513j, 0.9097+0.7468j,
        0.6124+0.5379j, 0.6493+0.7988j, 0.8082+0.8489j, 0.8252+0.6155j,
        0.7246+0.5871j, 0.8450+0.7259j, 0.6426+0.8164j, 0.6440+0.8479j,
        0.7326+0.6995j, 0.7463+0.6336j, 0.7983+0.6815j, 0.7503+0.6710j,
        0.7697+0.6189j, 0.6519+0.6634j, 0.6432+0.8146j, 0.6867+0

## modeling channel

In [16]:
ul_batch.shape

torch.Size([200, 2, 256, 32])

In [ ]:
import torch
import torch.nn as nn

class WirelessChannelSimulator(nn.Module):
    def __init__(self, num_bs_antennas=32):
        """
        Simulates the wireless channel for CSI feedback using actual Uplink CSI.
        """
        super().__init__()
        self.Nt = num_bs_antennas

    def forward(self, s, snr_db, h_uplink_raw):
        """
        Args:
            s (torch.Tensor): Complex symbols [B, K]. 
            snr_db (torch.Tensor): SNR in dB [B, 1].
            h_uplink_raw (torch.Tensor): Uplink CSI batch.
                                         Expected Shape: [B, 2, Nc, Nt] 
                                         Example: [200, 2, 256, 32]
        """
        batch_size, k = s.shape
        device = s.device

        # --- 1. Process Uplink Channel (H_u) ---
        # Input Shape: [B, 2, Nc, Nt] -> (Batch, Real/Imag, Subcarriers, Antennas)
        
        # Check if we have enough subcarriers (Axis 2)
        num_subcarriers = h_uplink_raw.shape[2]
        if num_subcarriers < k:
            raise ValueError(f"Uplink channel subcarriers ({num_subcarriers}) fewer than feedback symbols ({k})")
            
        # Slice the subcarriers (Axis 2) to match symbol length K
        # New Shape: [B, 2, K, Nt]
        h_sliced = h_uplink_raw[:, :, :k, :] 
        
        # Extract Real and Imaginary parts from Axis 1 (Channels)
        # h_real/h_imag Shape: [B, K, Nt]
        h_real = h_sliced[:, 0, :, :]
        h_imag = h_sliced[:, 1, :, :]
        
        # Combine into complex tensor: [B, K, Nt]
        h_u = torch.complex(h_real, h_imag)

        # --- 2. Generate AWGN Noise (z) ---
        # Calculate noise deviation based on SNR
        snr_linear = 10 ** (snr_db / 10.0) # [B, 1]
        noise_power = 1.0 / snr_linear
        noise_std = torch.sqrt(noise_power / 2.0)
        noise_std = noise_std.unsqueeze(-1) # [B, 1, 1] for broadcasting

        # Generate noise matching h_u dimensions [B, K, Nt]
        z_real = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z_imag = torch.randn(batch_size, k, self.Nt, device=device) * noise_std
        z = torch.complex(z_real, z_imag)

        # --- 3. Transmission ---
        # Expand s for broadcasting across antennas: [B, K] -> [B, K, 1]
        s_expanded = s.unsqueeze(-1)
        
        # Received signal y: [B, K, Nt]
        y = h_u * s_expanded + z

        # --- 4. Maximum Ratio Combining (MRC) ---
        # Calculate norm per subcarrier: [B, K, 1]
        h_norm = torch.norm(h_u, dim=2, keepdim=True)
        h_norm = torch.clamp(h_norm, min=1e-8)
        
        # Combining vector w: [B, K, Nt]
        w = h_u / h_norm

        # Apply MRC: sum(conj(w) * y) across antennas (dim=2)
        # Result: [B, K]
        s_hat = torch.sum(torch.conj(w) * y, dim=2)
        
        # --- 5. Normalization ---
        # Normalize by channel gain to recover scaled symbols for the neural decoder
        s_hat = s_hat / h_norm.squeeze(-1)

        return s_hat

In [ ]:
# --- Usage Example ---
if __name__ == "__main__":

    Nt = 32 
    
    input_symbols = complex_symbols
    snr_db = snr_batch

    # Initialize Simulator
    channel_sim = WirelessChannelSimulator(num_bs_antennas=Nt).to(device)

    # Simulate Transmission
    received_symbols = channel_sim(input_symbols.to(device), snr_db.to(device),ul_batch.to(device))

    print(f"Input Symbols Shape: {input_symbols.shape}") # [B, K]
    print(f"SNR Shape: {snr_db.shape}")                 # [B, 1]
    print(f"Output Symbols Shape: {received_symbols.shape}") # [B, K]
    
    

Input Symbols Shape: torch.Size([200, 64])
SNR Shape: torch.Size([200, 1])
Output Symbols Shape: torch.Size([200, 64])


## C2R

In [19]:
import torch
import torch.nn as nn

class ComplexToReal(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, s_hat):
        """
        Converts complex symbols to real-valued vector by concatenating 
        real and imaginary parts.
        
        Args:
            s_hat (torch.Tensor): Complex tensor of shape [B, K].
                                  Example: [B, 8]
        
        Returns:
            torch.Tensor: Real tensor of shape [B, 2*K].
                          Example: [B, 16]
                          Structure: [Real_1...Real_K, Imag_1...Imag_K]
        """
        # Extract real and imaginary parts
        # Shape of each: [B, K]
        real_part = s_hat.real
        imag_part = s_hat.imag
        
        # Concatenate along the feature dimension (dim=1)
        # Resulting shape: [B, K + K] -> [B, 2K]
        c_hat = torch.cat((real_part, imag_part), dim=1)
        
        return c_hat



In [20]:

if __name__ == "__main__":

    s_hat = received_symbols
    
    # Initialize the converter
    c2r = ComplexToReal()
    
    # Perform conversion
    c_hat = c2r(s_hat)
    
    print(f"Input Shape (Complex): {s_hat.shape}")  # [2, 8]
    print(f"Output Shape (Real):   {c_hat.shape}")  # [2, 16]


Input Shape (Complex): torch.Size([200, 64])
Output Shape (Real):   torch.Size([200, 128])


## Decoder

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ModifiedRefineNetBlock(nn.Module):
    """
    Modified RefineNet Block with AF Modules.
    Paper Rule: AF Module is placed after each main layer[cite: 348, 366].
    """
    def __init__(self, channels):
        super(ModifiedRefineNetBlock, self).__init__()
        
        # --- Layer 1 ---
        # Main Layer: Conv 7x7 -> BN -> LeakyReLU
        self.conv1 = nn.Conv2d(in_channels=channels, out_channels=8, 
                               kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm2d(8)
        # AF Module 1 (Input channels: 8)
        self.af1 = AFModule(8)
        
        # --- Layer 2 ---
        # Main Layer: Conv 5x5 -> BN -> LeakyReLU
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, 
                               kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        # AF Module 2 (Input channels: 16)
        self.af2 = AFModule(16)
        
        # --- Layer 3 ---
        # Main Layer: Conv 3x3 -> BN -> Tanh
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=channels, 
                               kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(channels)
        # AF Module 3 (Input channels: original 'channels', e.g., 2)
        self.af3 = AFModule(channels)
        
    def forward(self, x, snr):
        identity = x
        
        # Block 1
        out = self.conv1(x)
        out = self.bn1(out)
        out = F.leaky_relu(out, negative_slope=0.3)
        out = self.af1(out, snr) # Apply AF
        
        # Block 2
        out = self.conv2(out)
        out = self.bn2(out)
        out = F.leaky_relu(out, negative_slope=0.3)
        out = self.af2(out, snr) # Apply AF
        
        # Block 3
        out = self.conv3(out)
        out = self.bn3(out)
        out = torch.tanh(out)
        out = self.af3(out, snr) # Apply AF
        
        # Residual
        out = identity + out
        out = F.relu(out)
        
        return out

class CsiNetPlusDecoder(nn.Module):
    def __init__(self, input_dim=128, height=32, width=32, channels=2, num_blocks=5):
        super(CsiNetPlusDecoder, self).__init__()
        
        self.height = height
        self.width = width
        self.channels = channels
        self.flattened_dim = height * width * channels 
        
        # 1. FC Layer
        self.fc = nn.Linear(input_dim, self.flattened_dim)
        
        # 2. Initial Refinement
        self.initial_conv = nn.Conv2d(in_channels=channels, out_channels=channels, 
                                      kernel_size=7, padding=3)
        self.initial_bn = nn.BatchNorm2d(channels)
        
        # AF Module for Initial Refinement
        self.initial_af = AFModule(channels)
        
        # 3. RefineNet Chain
        # Use ModuleList instead of Sequential to pass 'snr' in the loop
        self.refinenet_chain = nn.ModuleList(
            [ModifiedRefineNetBlock(channels) for _ in range(num_blocks)]
        )
        
    def forward(self, x, snr):
        """
        Args:
            x: Input vector [Batch, input_dim]
            snr: SNR vector [Batch, 1]
        """
        # FC Layer
        x = self.fc(x)
        x = x.view(-1, self.channels, self.height, self.width)
        
        # Initial Refinement with AF
        x = self.initial_conv(x)
        x = self.initial_bn(x)
        x = torch.sigmoid(x)
        x = self.initial_af(x, snr) # Apply AF [cite: 366]
        
        # RefineNet Chain with AF (Passing SNR to each block)
        for block in self.refinenet_chain:
            x = block(x, snr)
        
        return x

In [22]:

model = CsiNetPlusDecoder().to(device)

# 3. Create Dummy Data
# x: The compressed codeword vector
x = c_hat

# snr: The Signal-to-Noise Ratio for each sample in the batch
# Shape must be (Batch, 1) as expected by AFModule
snr = snr_batch

# 4. Forward Pass
output_decoder = model(x.to(device), snr.to(device))

# 5. Check Output
print("Input shape: ", x.shape)
print("SNR shape:   ", snr.shape)
print("Output shape:", output_decoder.shape) 
# Expected Output shape: (32, 2, 32, 2) matches (Batch, Channels, Height, Width)

Input shape:  torch.Size([200, 128])
SNR shape:    torch.Size([200, 1])
Output shape: torch.Size([200, 2, 32, 32])


## STN

In [23]:
class STN(nn.Module):
    """
    Synthesis Transform Network (STN) as described in Fig. 6(b) and Section V.C [cite: 2771, 2807-2809].
    
    Structure:
    1. TransConv (16 filters, 3x3, stride 2x1) -> BN -> PReLU -> AFModule
    2. TransConv (6 filters, 3x3, stride 2x1) -> BN -> PReLU -> AFModule
    3. TransConv (2 filters, 3x3, stride 2x1) -> BN -> Sigmoid
    
    Target: Upsample from (32, 32) -> (256, 32) (Subcarriers x Antennas)
    This requires 3 upsampling steps of factor 2 on the height (subcarrier) axis only.
    """
    def __init__(self):
        super(STN, self).__init__()
        
        # --- Layer 1 ---
        # Input: (B, 2, 32, 32) -> Upsample H by 2 -> (B, 16, 64, 32)
        # Paper Fig 6b: "TransConv 16 | (3,3) | (2,1) ^" 
        # (16 filters, 3x3 kernel, stride 2 vertical, 1 horizontal)
        self.trans_conv1 = nn.ConvTranspose2d(
            in_channels=2, out_channels=16, 
            kernel_size=(3, 3), stride=(2, 1), 
            padding=(1, 1), output_padding=(1, 0)
        )
        self.bn1 = nn.BatchNorm2d(16)
        self.prelu1 = nn.PReLU(num_parameters=16) # [cite: 2802]
        self.af1 = AFModule(channels=16)          # [cite: 2771]

        # --- Layer 2 ---
        # Input: (B, 16, 64, 32) -> Upsample H by 2 -> (B, 6, 128, 32)
        # Paper Fig 6b: "TransConv 6 | (3,3) | (2,1) ^"
        self.trans_conv2 = nn.ConvTranspose2d(
            in_channels=16, out_channels=6, 
            kernel_size=(3, 3), stride=(2, 1), 
            padding=(1, 1), output_padding=(1, 0)
        )
        self.bn2 = nn.BatchNorm2d(6)
        self.prelu2 = nn.PReLU(num_parameters=6)
        self.af2 = AFModule(channels=6)

        # --- Layer 3 ---
        # Input: (B, 6, 128, 32) -> Upsample H by 2 -> (B, 2, 256, 32)
        # Paper Fig 6b: "TransConv 2 | (3,3) | (2,1) ^"
        self.trans_conv3 = nn.ConvTranspose2d(
            in_channels=6, out_channels=2, 
            kernel_size=(3, 3), stride=(2, 1), 
            padding=(1, 1), output_padding=(1, 0)
        )
        self.bn3 = nn.BatchNorm2d(2) # Paper shows BN before Sigmoid in Fig 6b
        self.sigmoid = nn.Sigmoid()

    def forward(self, x, snr):
        """
        Args:
            x: Transformed CSI [B, 2, 32, 32]
            snr: Feedback SNR [B, 1]
        Returns:
            out: Recovered CSI [B, 2, 256, 32]
        """
        # Block 1
        out = self.trans_conv1(x)
        out = self.bn1(out)
        out = self.prelu1(out)
        out = self.af1(out, snr)

        # Block 2
        out = self.trans_conv2(out)
        out = self.bn2(out)
        out = self.prelu2(out)
        out = self.af2(out, snr)

        # Block 3
        out = self.trans_conv3(out)
        out = self.bn3(out)
        out = self.sigmoid(out)

        return out



In [24]:

if __name__ == "__main__":
    batch_size = 8
    # Input shape: (B, 2, 32, 2)
    input_tensor = output_decoder
    snr_tensor = snr_batch

    model = STN().to(device)
    output = model(input_tensor.to(device), snr_tensor.to(device))

    print("Input Shape:", input_tensor.shape)   # Expected: (8, 2, 32, 2)
    print("Output Shape:", output.shape) # Expected: (8, 13, 32, 2)

Input Shape: torch.Size([200, 2, 32, 32])
Output Shape: torch.Size([200, 2, 256, 32])


### flop counts

In [25]:
"""
FLOP Counter for ADJSCC-CSInet Model
Calculates the computational complexity (FLOPs) for the entire model including:
- ATN (Analysis Transform Network)
- Encoder (Joint Source-Channel Encoder with AF Modules)
- Decoder (Joint Source-Channel Decoder with AF Modules)
- STN (Synthesis Transform Network)
"""


def count_conv2d_flops(in_channels, out_channels, kernel_size, input_shape, stride=1, padding=0):
    """
    Calculate FLOPs for a 2D convolution layer.
    
    FLOPs = 2 * H_out * W_out * C_in * K_h * K_w * C_out
    (factor of 2 accounts for multiply-add)
    
    Args:
        in_channels: Input channels
        out_channels: Output channels
        kernel_size: Kernel size (h, w) or scalar
        input_shape: (H_in, W_in)
        stride: Stride (h, w) or scalar
        padding: Padding (h, w) or scalar
    """
    if isinstance(kernel_size, int):
        k_h, k_w = kernel_size, kernel_size
    else:
        k_h, k_w = kernel_size
    
    if isinstance(stride, int):
        s_h, s_w = stride, stride
    else:
        s_h, s_w = stride
    
    if isinstance(padding, int):
        p_h, p_w = padding, padding
    else:
        p_h, p_w = padding
    
    h_in, w_in = input_shape
    
    # Calculate output dimensions
    h_out = (h_in + 2 * p_h - k_h) // s_h + 1
    w_out = (w_in + 2 * p_w - k_w) // s_w + 1
    
    # FLOPs = 2 (for MAC) * output_spatial * input_channels * kernel_spatial * output_channels
    flops = 2 * h_out * w_out * in_channels * k_h * k_w * out_channels
    
    return flops, (h_out, w_out)


def count_conv_transpose2d_flops(in_channels, out_channels, kernel_size, input_shape, stride=1, padding=0, output_padding=0):
    """
    Calculate FLOPs for a 2D transposed convolution layer.
    
    Args:
        in_channels: Input channels
        out_channels: Output channels
        kernel_size: Kernel size (h, w) or scalar
        input_shape: (H_in, W_in)
        stride: Stride (h, w) or scalar
        padding: Padding (h, w) or scalar
        output_padding: Output padding (h, w) or scalar
    """
    if isinstance(kernel_size, int):
        k_h, k_w = kernel_size, kernel_size
    else:
        k_h, k_w = kernel_size
    
    if isinstance(stride, int):
        s_h, s_w = stride, stride
    else:
        s_h, s_w = stride
    
    if isinstance(padding, int):
        p_h, p_w = padding, padding
    else:
        p_h, p_w = padding
        
    if isinstance(output_padding, int):
        op_h, op_w = output_padding, output_padding
    else:
        op_h, op_w = output_padding
    
    h_in, w_in = input_shape
    
    # Calculate output dimensions for transposed conv
    h_out = (h_in - 1) * s_h - 2 * p_h + k_h + op_h
    w_out = (w_in - 1) * s_w - 2 * p_w + k_w + op_w
    
    # FLOPs = 2 * output_spatial * input_channels * kernel_spatial * output_channels
    flops = 2 * h_out * w_out * in_channels * k_h * k_w * out_channels
    
    return flops, (h_out, w_out)


def count_linear_flops(in_features, out_features):
    """
    Calculate FLOPs for a fully connected layer.
    FLOPs = 2 * in_features * out_features (multiply-add)
    """
    return 2 * in_features * out_features


def count_batchnorm_flops(num_features, spatial_size):
    """
    Calculate FLOPs for batch normalization.
    BN requires: mean, variance, normalize, scale, shift
    Approximate: 2 * num_features * spatial_size
    """
    h, w = spatial_size
    return 2 * num_features * h * w


def count_activation_flops(spatial_size, channels):
    """
    Calculate FLOPs for activation functions (ReLU, PReLU, Sigmoid, etc.)
    Approximate: 1 FLOP per element
    """
    h, w = spatial_size
    return channels * h * w


def count_af_module_flops(channels, spatial_size, reduction_ratio=2):
    """
    Calculate FLOPs for the AF (Attention Feature) Module.
    
    Components:
    1. Global Average Pooling: channels * H * W additions
    2. FC1: Linear(channels + 1, hidden_dim)
    3. ReLU: hidden_dim
    4. FC2: Linear(hidden_dim, channels)
    5. Sigmoid: channels
    6. Element-wise multiplication: channels * H * W
    """
    h, w = spatial_size
    hidden_dim = max(channels // reduction_ratio, 1)
    
    # Global average pooling
    gap_flops = channels * h * w
    
    # FC layers
    fc1_flops = count_linear_flops(channels + 1, hidden_dim)
    fc2_flops = count_linear_flops(hidden_dim, channels)
    
    # Activations
    relu_flops = hidden_dim
    sigmoid_flops = channels
    
    # Element-wise multiplication (recalibration)
    mult_flops = channels * h * w
    
    total = gap_flops + fc1_flops + relu_flops + fc2_flops + sigmoid_flops + mult_flops
    return total


def count_atn_flops(input_shape=(256, 32), input_channels=2):
    """
    Count FLOPs for ATN (Analysis Transform Network).
    
    Architecture:
    Block 1: Conv2d(2->16, k=3x3, s=2x1, p=1x1) -> BN -> PReLU -> AF
    Block 2: Conv2d(16->16, k=3x3, s=2x1, p=1x1) -> BN -> PReLU -> AF
    Block 3: Conv2d(16->2, k=3x3, s=2x1, p=1x1) -> BN -> Sigmoid
    """
    total_flops = 0
    h, w = input_shape
    
    print("\n=== ATN (Analysis Transform Network) ===")
    
    # Block 1
    print("\nBlock 1:")
    conv1_flops, (h1, w1) = count_conv2d_flops(2, 16, (3, 3), (h, w), stride=(2, 1), padding=(1, 1))
    bn1_flops = count_batchnorm_flops(16, (h1, w1))
    prelu1_flops = count_activation_flops((h1, w1), 16)
    af1_flops = count_af_module_flops(16, (h1, w1))
    
    block1_flops = conv1_flops + bn1_flops + prelu1_flops + af1_flops
    print(f"  Conv1: {conv1_flops:,} FLOPs, Output: ({h1}, {w1})")
    print(f"  BN1: {bn1_flops:,} FLOPs")
    print(f"  PReLU1: {prelu1_flops:,} FLOPs")
    print(f"  AF1: {af1_flops:,} FLOPs")
    print(f"  Block 1 Total: {block1_flops:,} FLOPs")
    total_flops += block1_flops
    
    # Block 2
    print("\nBlock 2:")
    conv2_flops, (h2, w2) = count_conv2d_flops(16, 16, (3, 3), (h1, w1), stride=(2, 1), padding=(1, 1))
    bn2_flops = count_batchnorm_flops(16, (h2, w2))
    prelu2_flops = count_activation_flops((h2, w2), 16)
    af2_flops = count_af_module_flops(16, (h2, w2))
    
    block2_flops = conv2_flops + bn2_flops + prelu2_flops + af2_flops
    print(f"  Conv2: {conv2_flops:,} FLOPs, Output: ({h2}, {w2})")
    print(f"  BN2: {bn2_flops:,} FLOPs")
    print(f"  PReLU2: {prelu2_flops:,} FLOPs")
    print(f"  AF2: {af2_flops:,} FLOPs")
    print(f"  Block 2 Total: {block2_flops:,} FLOPs")
    total_flops += block2_flops
    
    # Block 3
    print("\nBlock 3:")
    conv3_flops, (h3, w3) = count_conv2d_flops(16, 2, (3, 3), (h2, w2), stride=(2, 1), padding=(1, 1))
    bn3_flops = count_batchnorm_flops(2, (h3, w3))
    sigmoid_flops = count_activation_flops((h3, w3), 2)
    
    block3_flops = conv3_flops + bn3_flops + sigmoid_flops
    print(f"  Conv3: {conv3_flops:,} FLOPs, Output: ({h3}, {w3})")
    print(f"  BN3: {bn3_flops:,} FLOPs")
    print(f"  Sigmoid: {sigmoid_flops:,} FLOPs")
    print(f"  Block 3 Total: {block3_flops:,} FLOPs")
    total_flops += block3_flops
    
    print(f"\n*** ATN Total: {total_flops:,} FLOPs ***")
    return total_flops, (h3, w3)


def count_encoder_flops(input_shape=(32, 32), input_channels=2, compression_ratio=16):
    """
    Count FLOPs for the Encoder (CsiNetPlusEncoderWithAF).
    
    Architecture:
    Conv1(2->2, k=7x7, s=1, p=3) -> BN -> LeakyReLU -> AF
    Conv2(2->2, k=7x7, s=1, p=3) -> BN -> LeakyReLU -> AF
    Flatten -> FC(2048 -> 128) -> Sigmoid
    """
    total_flops = 0
    h, w = input_shape
    total_elements = input_channels * h * w
    M = total_elements // compression_ratio  # 2048 / 16 = 128
    
    print("\n=== Encoder (Joint Source-Channel Encoder) ===")
    
    # Block 1
    print("\nBlock 1:")
    conv1_flops, (h1, w1) = count_conv2d_flops(2, 2, 7, (h, w), stride=1, padding=3)
    bn1_flops = count_batchnorm_flops(2, (h1, w1))
    act1_flops = count_activation_flops((h1, w1), 2)
    af1_flops = count_af_module_flops(2, (h1, w1))
    
    block1_flops = conv1_flops + bn1_flops + act1_flops + af1_flops
    print(f"  Conv1: {conv1_flops:,} FLOPs, Output: ({h1}, {w1})")
    print(f"  BN1: {bn1_flops:,} FLOPs")
    print(f"  LeakyReLU1: {act1_flops:,} FLOPs")
    print(f"  AF1: {af1_flops:,} FLOPs")
    print(f"  Block 1 Total: {block1_flops:,} FLOPs")
    total_flops += block1_flops
    
    # Block 2
    print("\nBlock 2:")
    conv2_flops, (h2, w2) = count_conv2d_flops(2, 2, 7, (h1, w1), stride=1, padding=3)
    bn2_flops = count_batchnorm_flops(2, (h2, w2))
    act2_flops = count_activation_flops((h2, w2), 2)
    af2_flops = count_af_module_flops(2, (h2, w2))
    
    block2_flops = conv2_flops + bn2_flops + act2_flops + af2_flops
    print(f"  Conv2: {conv2_flops:,} FLOPs, Output: ({h2}, {w2})")
    print(f"  BN2: {bn2_flops:,} FLOPs")
    print(f"  LeakyReLU2: {act2_flops:,} FLOPs")
    print(f"  AF2: {af2_flops:,} FLOPs")
    print(f"  Block 2 Total: {block2_flops:,} FLOPs")
    total_flops += block2_flops
    
    # Fully Connected Layer
    print("\nFC Layer:")
    fc_flops = count_linear_flops(total_elements, M)
    sigmoid_flops = M
    
    fc_total = fc_flops + sigmoid_flops
    print(f"  FC({total_elements} -> {M}): {fc_flops:,} FLOPs")
    print(f"  Sigmoid: {sigmoid_flops:,} FLOPs")
    print(f"  FC Total: {fc_total:,} FLOPs")
    total_flops += fc_total
    
    print(f"\n*** Encoder Total: {total_flops:,} FLOPs ***")
    return total_flops


def count_refinenet_block_flops(channels, input_shape):
    """
    Count FLOPs for a single RefineNet block.
    
    Architecture:
    Conv1(C->8, k=7x7, p=3) -> BN -> LeakyReLU -> AF
    Conv2(8->16, k=5x5, p=2) -> BN -> LeakyReLU -> AF
    Conv3(16->C, k=3x3, p=1) -> BN -> Tanh -> AF
    Residual add + ReLU
    """
    total_flops = 0
    h, w = input_shape
    
    # Layer 1
    conv1_flops, (h1, w1) = count_conv2d_flops(channels, 8, 7, (h, w), stride=1, padding=3)
    bn1_flops = count_batchnorm_flops(8, (h1, w1))
    act1_flops = count_activation_flops((h1, w1), 8)
    af1_flops = count_af_module_flops(8, (h1, w1))
    layer1_flops = conv1_flops + bn1_flops + act1_flops + af1_flops
    
    # Layer 2
    conv2_flops, (h2, w2) = count_conv2d_flops(8, 16, 5, (h1, w1), stride=1, padding=2)
    bn2_flops = count_batchnorm_flops(16, (h2, w2))
    act2_flops = count_activation_flops((h2, w2), 16)
    af2_flops = count_af_module_flops(16, (h2, w2))
    layer2_flops = conv2_flops + bn2_flops + act2_flops + af2_flops
    
    # Layer 3
    conv3_flops, (h3, w3) = count_conv2d_flops(16, channels, 3, (h2, w2), stride=1, padding=1)
    bn3_flops = count_batchnorm_flops(channels, (h3, w3))
    tanh_flops = count_activation_flops((h3, w3), channels)
    af3_flops = count_af_module_flops(channels, (h3, w3))
    layer3_flops = conv3_flops + bn3_flops + tanh_flops + af3_flops
    
    # Residual addition and ReLU
    add_flops = channels * h * w
    relu_flops = channels * h * w
    
    total_flops = layer1_flops + layer2_flops + layer3_flops + add_flops + relu_flops
    return total_flops


def count_decoder_flops(input_dim=128, output_shape=(32, 32), channels=2, num_blocks=5):
    """
    Count FLOPs for the Decoder (CsiNetPlusDecoder).
    
    Architecture:
    FC(128 -> 2048) -> Reshape
    Initial Conv(2->2, k=7x7, p=3) -> BN -> Sigmoid -> AF
    5x RefineNet blocks
    """
    total_flops = 0
    h, w = output_shape
    flattened_dim = h * w * channels
    
    print("\n=== Decoder (Joint Source-Channel Decoder) ===")
    
    # FC Layer
    print("\nFC Layer:")
    fc_flops = count_linear_flops(input_dim, flattened_dim)
    print(f"  FC({input_dim} -> {flattened_dim}): {fc_flops:,} FLOPs")
    total_flops += fc_flops
    
    # Initial Refinement
    print("\nInitial Refinement:")
    conv_flops, (h_out, w_out) = count_conv2d_flops(channels, channels, 7, (h, w), stride=1, padding=3)
    bn_flops = count_batchnorm_flops(channels, (h_out, w_out))
    sigmoid_flops = count_activation_flops((h_out, w_out), channels)
    af_flops = count_af_module_flops(channels, (h_out, w_out))
    
    initial_flops = conv_flops + bn_flops + sigmoid_flops + af_flops
    print(f"  Conv: {conv_flops:,} FLOPs")
    print(f"  BN: {bn_flops:,} FLOPs")
    print(f"  Sigmoid: {sigmoid_flops:,} FLOPs")
    print(f"  AF: {af_flops:,} FLOPs")
    print(f"  Initial Total: {initial_flops:,} FLOPs")
    total_flops += initial_flops
    
    # RefineNet Blocks
    print(f"\nRefineNet Blocks (x{num_blocks}):")
    block_flops = count_refinenet_block_flops(channels, (h_out, w_out))
    total_refinenet_flops = num_blocks * block_flops
    print(f"  Single Block: {block_flops:,} FLOPs")
    print(f"  Total ({num_blocks} blocks): {total_refinenet_flops:,} FLOPs")
    total_flops += total_refinenet_flops
    
    print(f"\n*** Decoder Total: {total_flops:,} FLOPs ***")
    return total_flops


def count_stn_flops(input_shape=(32, 32), input_channels=2):
    """
    Count FLOPs for STN (Synthesis Transform Network).
    
    Architecture:
    TransConv1(2->16, k=3x3, s=2x1, p=1x1, op=1x0) -> BN -> PReLU -> AF
    TransConv2(16->6, k=3x3, s=2x1, p=1x1, op=1x0) -> BN -> PReLU -> AF
    TransConv3(6->2, k=3x3, s=2x1, p=1x1, op=1x0) -> BN -> Sigmoid
    """
    total_flops = 0
    h, w = input_shape
    
    print("\n=== STN (Synthesis Transform Network) ===")
    
    # Layer 1
    print("\nLayer 1:")
    tconv1_flops, (h1, w1) = count_conv_transpose2d_flops(
        2, 16, (3, 3), (h, w), stride=(2, 1), padding=(1, 1), output_padding=(1, 0)
    )
    bn1_flops = count_batchnorm_flops(16, (h1, w1))
    prelu1_flops = count_activation_flops((h1, w1), 16)
    af1_flops = count_af_module_flops(16, (h1, w1))
    
    layer1_flops = tconv1_flops + bn1_flops + prelu1_flops + af1_flops
    print(f"  TransConv1: {tconv1_flops:,} FLOPs, Output: ({h1}, {w1})")
    print(f"  BN1: {bn1_flops:,} FLOPs")
    print(f"  PReLU1: {prelu1_flops:,} FLOPs")
    print(f"  AF1: {af1_flops:,} FLOPs")
    print(f"  Layer 1 Total: {layer1_flops:,} FLOPs")
    total_flops += layer1_flops
    
    # Layer 2
    print("\nLayer 2:")
    tconv2_flops, (h2, w2) = count_conv_transpose2d_flops(
        16, 6, (3, 3), (h1, w1), stride=(2, 1), padding=(1, 1), output_padding=(1, 0)
    )
    bn2_flops = count_batchnorm_flops(6, (h2, w2))
    prelu2_flops = count_activation_flops((h2, w2), 6)
    af2_flops = count_af_module_flops(6, (h2, w2))
    
    layer2_flops = tconv2_flops + bn2_flops + prelu2_flops + af2_flops
    print(f"  TransConv2: {tconv2_flops:,} FLOPs, Output: ({h2}, {w2})")
    print(f"  BN2: {bn2_flops:,} FLOPs")
    print(f"  PReLU2: {prelu2_flops:,} FLOPs")
    print(f"  AF2: {af2_flops:,} FLOPs")
    print(f"  Layer 2 Total: {layer2_flops:,} FLOPs")
    total_flops += layer2_flops
    
    # Layer 3
    print("\nLayer 3:")
    tconv3_flops, (h3, w3) = count_conv_transpose2d_flops(
        6, 2, (3, 3), (h2, w2), stride=(2, 1), padding=(1, 1), output_padding=(1, 0)
    )
    bn3_flops = count_batchnorm_flops(2, (h3, w3))
    sigmoid_flops = count_activation_flops((h3, w3), 2)
    
    layer3_flops = tconv3_flops + bn3_flops + sigmoid_flops
    print(f"  TransConv3: {tconv3_flops:,} FLOPs, Output: ({h3}, {w3})")
    print(f"  BN3: {bn3_flops:,} FLOPs")
    print(f"  Sigmoid: {sigmoid_flops:,} FLOPs")
    print(f"  Layer 3 Total: {layer3_flops:,} FLOPs")
    total_flops += layer3_flops
    
    print(f"\n*** STN Total: {total_flops:,} FLOPs ***")
    return total_flops, (h3, w3)


def count_total_model_flops():
    """
    Count total FLOPs for the entire ADJSCC-CSInet model.
    
    Complete pipeline:
    1. ATN: (B, 2, 256, 32) -> (B, 2, 32, 32)
    2. Encoder: (B, 2, 32, 32) -> (B, 128)
    3. R2C + Power Normalization: (B, 128) -> (B, 64) complex
    4. Channel Simulation (not counted, it's wireless channel)
    5. C2R: (B, 64) complex -> (B, 128)
    6. Decoder: (B, 128) -> (B, 2, 32, 32)
    7. STN: (B, 2, 32, 32) -> (B, 2, 256, 32)
    """
    print("="*80)
    print("FLOP COUNT FOR ADJSCC-CSINET MODEL")
    print("="*80)
    
    total_flops = 0
    
    # ATN
    atn_flops, atn_output_shape = count_atn_flops(input_shape=(256, 32), input_channels=2)
    total_flops += atn_flops
    
    # Encoder
    encoder_flops = count_encoder_flops(input_shape=atn_output_shape, input_channels=2, compression_ratio=16)
    total_flops += encoder_flops
    
    # R2C and Power Normalization (minimal FLOPs, mainly data transformation)
    # We'll count basic operations: 128 elements -> 64 complex (no major compute)
    r2c_flops = 128  # Approximate for splitting and complex formation
    power_norm_flops = 64 * 4  # Approximate: abs, square, mean, sqrt, divide
    r2c_total = r2c_flops + power_norm_flops
    print(f"\n=== R2C & Power Normalization ===")
    print(f"R2C: {r2c_flops:,} FLOPs")
    print(f"Power Normalization: {power_norm_flops:,} FLOPs")
    print(f"*** Total: {r2c_total:,} FLOPs ***")
    total_flops += r2c_total
    
    # Channel Simulation - NOT COUNTED (wireless channel, not model computation)
    print(f"\n=== Channel Simulation ===")
    print("Channel simulation FLOPs not counted (wireless channel effects)")
    
    # C2R (Complex to Real)
    c2r_flops = 64 * 2  # Extract real and imag parts, concat
    print(f"\n=== C2R (Complex to Real) ===")
    print(f"*** Total: {c2r_flops:,} FLOPs ***")
    total_flops += c2r_flops
    
    # Decoder
    decoder_flops = count_decoder_flops(input_dim=128, output_shape=(32, 32), channels=2, num_blocks=5)
    total_flops += decoder_flops
    
    # STN
    stn_flops, stn_output_shape = count_stn_flops(input_shape=(32, 32), input_channels=2)
    total_flops += stn_flops
    
    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"ATN FLOPs:              {atn_flops:>15,}")
    print(f"Encoder FLOPs:          {encoder_flops:>15,}")
    print(f"R2C & Norm FLOPs:       {r2c_total:>15,}")
    print(f"C2R FLOPs:              {c2r_flops:>15,}")
    print(f"Decoder FLOPs:          {decoder_flops:>15,}")
    print(f"STN FLOPs:              {stn_flops:>15,}")
    print("-"*80)
    print(f"TOTAL MODEL FLOPs:      {total_flops:>15,}")
    print(f"TOTAL (MFLOPs):         {total_flops/1e6:>15.2f}")
    print(f"TOTAL (GFLOPs):         {total_flops/1e9:>15.4f}")
    print("="*80)
    
    return total_flops


if __name__ == "__main__":
    total = count_total_model_flops()

FLOP COUNT FOR ADJSCC-CSINET MODEL

=== ATN (Analysis Transform Network) ===

Block 1:
  Conv1: 2,359,296 FLOPs, Output: (128, 32)
  BN1: 131,072 FLOPs
  PReLU1: 65,536 FLOPs
  AF1: 131,624 FLOPs
  Block 1 Total: 2,687,528 FLOPs

Block 2:
  Conv2: 9,437,184 FLOPs, Output: (64, 32)
  BN2: 65,536 FLOPs
  PReLU2: 32,768 FLOPs
  AF2: 66,088 FLOPs
  Block 2 Total: 9,601,576 FLOPs

Block 3:
  Conv3: 589,824 FLOPs, Output: (32, 32)
  BN3: 4,096 FLOPs
  Sigmoid: 2,048 FLOPs
  Block 3 Total: 595,968 FLOPs

*** ATN Total: 12,885,072 FLOPs ***

=== Encoder (Joint Source-Channel Encoder) ===

Block 1:
  Conv1: 401,408 FLOPs, Output: (32, 32)
  BN1: 4,096 FLOPs
  LeakyReLU1: 2,048 FLOPs
  AF1: 4,109 FLOPs
  Block 1 Total: 411,661 FLOPs

Block 2:
  Conv2: 401,408 FLOPs, Output: (32, 32)
  BN2: 4,096 FLOPs
  LeakyReLU2: 2,048 FLOPs
  AF2: 4,109 FLOPs
  Block 2 Total: 411,661 FLOPs

FC Layer:
  FC(2048 -> 128): 524,288 FLOPs
  Sigmoid: 128 FLOPs
  FC Total: 524,416 FLOPs

*** Encoder Total: 1,347,738 

## training loop

In [26]:
device

device(type='cuda')

In [27]:
# Initialize your implemented modules
atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=16).to(device)
channel_sim = WirelessChannelSimulator(num_bs_antennas=32).to(device)
c2r = ComplexToReal().to(device)
decoder = CsiNetPlusDecoder().to(device)
stn = STN().to(device)

# Combine parameters for global end-to-end training [cite: 93, 396]
all_params = (list(atn.parameters()) + list(encoder.parameters()) + 
              list(decoder.parameters()) + list(stn.parameters()))

In [28]:
# Count total trainable parameters across all modules
total_params = sum(p.numel() for p in all_params if p.requires_grad)

print(f"Total Trainable Parameters: {total_params:,}")
print(f"Total Trainable Parameters (Millions): {total_params / 1e6:.2f}M")

Total Trainable Parameters: 555,974
Total Trainable Parameters (Millions): 0.56M


In [30]:
import torch
import torch.optim as optim
import torch.nn as nn
import numpy as np
import time  # Added for timing

# [cite_start]--- 1. Hyperparameters [cite: 433-436] ---
BATCH_SIZE = 200
LEARNING_RATE = 1e-3
MIN_LR = 1e-5
EPOCHS = 500
PATIENCE = 20
TRAIN_SAMPLES = 80000
VAL_SAMPLES = 30000

# Number of iterations per epoch (based on dataset size)
TRAIN_ITERATIONS = TRAIN_SAMPLES // BATCH_SIZE
VAL_ITERATIONS = VAL_SAMPLES // BATCH_SIZE

# --- 2. Setup Optimizer and Scheduler ---
# [cite_start]Combine all trainable parameters [cite: 322]
# [cite_start]The paper uses End-to-End training optimizing {alpha, theta, phi, beta, gamma, psi, rho, tau} [cite: 368]
all_params = (list(atn.parameters()) + 
              list(encoder.parameters()) + 
              list(decoder.parameters()) + 
              list(stn.parameters()))

optimizer = optim.Adam(all_params, lr=LEARNING_RATE)

# [cite_start]Scheduler: Decay by half if loss stops decreasing [cite: 434]
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',   
    factor=0.5, 
    patience=PATIENCE, 
    min_lr=MIN_LR
)

criterion = nn.MSELoss()

def nmse_loss(H_true, H_pred):
    """
    [cite_start]Normalized Mean Square Error (NMSE) calculation[cite: 441].
    NMSE = E[ ||H - H_hat||^2 / ||H||^2 ]
    """
    mse = torch.sum((H_true - H_pred)**2, dim=[1, 2, 3])
    power = torch.sum(H_true**2, dim=[1, 2, 3])
    nmse = mse / power
    return 10 * torch.log10(torch.mean(nmse))

# --- 3. Training Loop ---
best_val_loss = float('inf')

print(f"Starting training for {EPOCHS} epochs...")
print(f"Training on device: {device}")

for epoch in range(EPOCHS):
    start_time = time.time()  # Start timer for the epoch
    
    # --- Training Phase ---
    atn.train()
    encoder.train()
    decoder.train()
    stn.train()
    
    total_train_loss = 0.0
    
    for i in range(TRAIN_ITERATIONS):
        # [cite_start]1. Get Batch [cite: 446] (Uniform SNR distribution handled by collater)
        # collater returns: (dl_batch, ul_batch, snr_batch)
        H_d, H_u, snr = collater(mode="train")
        
        # Move to device
        H_d = H_d.to(device)
        H_u = H_u.to(device)
        snr = snr.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # --- Forward Pass (End-to-End) ---
        # [cite_start]1. Non-linear Transform (UE): Spatial-Freq -> Transform Domain [cite: 260]
        T = atn(H_d, snr)
        
        # [cite_start]2. Joint Source-Channel Encoder (UE) [cite: 352]
        c = encoder(T, snr)
        
        # [cite_start]3. Power Normalization & R2C (UE) [cite: 324, 328]
        s = enc_to_complex_and_normalize(c)
        
        # [cite_start]4. Wireless Channel Simulation [cite: 192, 197]
        # Simulates Fading, Noise, and MRC
        s_hat = channel_sim(s, snr, H_u)
        
        # [cite_start]5. C2R Conversion (BS) [cite: 332]
        c_hat = c2r(s_hat)
        
        # [cite_start]6. Joint Source-Channel Decoder (BS) [cite: 363]
        T_hat = decoder(c_hat, snr)
        
        # [cite_start]7. Synthesis Transform (BS): Transform Domain -> Spatial-Freq [cite: 363]
        H_hat = stn(T_hat, snr)
        
        # --- Loss & Backprop ---
        # [cite_start]Optimize MSE [cite: 488, 370]
        loss = criterion(H_hat, H_d)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / TRAIN_ITERATIONS

    # --- Validation Phase ---
    atn.eval()
    encoder.eval()
    decoder.eval()
    stn.eval()
    
    total_val_loss = 0.0
    total_val_nmse = 0.0
    
    with torch.no_grad():
        for i in range(VAL_ITERATIONS):
            H_d_val, H_u_val, snr_val = collater(mode="val")
            
            H_d_val = H_d_val.to(device)
            H_u_val = H_u_val.to(device)
            snr_val = snr_val.to(device)
            
            # Forward Pass
            T_val = atn(H_d_val, snr_val)
            c_val = encoder(T_val, snr_val)
            s_val = enc_to_complex_and_normalize(c_val)
            s_hat_val = channel_sim(s_val, snr_val, H_u_val)
            c_hat_val = c2r(s_hat_val)
            T_hat_val = decoder(c_hat_val, snr_val)
            H_hat_val = stn(T_hat_val, snr_val)
            
            # Metrics
            loss_val = criterion(H_hat_val, H_d_val)
            nmse_val = nmse_loss(H_d_val, H_hat_val)
            
            total_val_loss += loss_val.item()
            total_val_nmse += nmse_val.item()

    avg_val_loss = total_val_loss / VAL_ITERATIONS
    avg_val_nmse = total_val_nmse / VAL_ITERATIONS
    
    # --- End Timer ---
    end_time = time.time()
    epoch_duration = end_time - start_time
    
    # --- Logging & Scheduler ---
    # [cite_start]Update Learning Rate based on Validation Loss [cite: 434]
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(avg_val_loss)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Time: {epoch_duration:.2f}s | "  # Added time print
          f"LR: {current_lr:.1e} | "
          f"Train Loss: {avg_train_loss:.6f} | "
          f"Val Loss: {avg_val_loss:.6f} | "
          f"Val NMSE: {avg_val_nmse:.2f} dB")

print("Training Complete.")

Starting training for 500 epochs...
Training on device: cuda
Epoch [1/500] | Time: 510.70s | LR: 1.0e-03 | Train Loss: 0.211679 | Val Loss: 0.163278 | Val NMSE: 22.05 dB
Epoch [2/500] | Time: 515.55s | LR: 1.0e-03 | Train Loss: 0.140335 | Val Loss: 0.110023 | Val NMSE: 20.34 dB
Epoch [3/500] | Time: 518.99s | LR: 1.0e-03 | Train Loss: 0.096781 | Val Loss: 0.080384 | Val NMSE: 18.97 dB
Epoch [4/500] | Time: 520.15s | LR: 1.0e-03 | Train Loss: 0.069211 | Val Loss: 0.055384 | Val NMSE: 17.35 dB
Epoch [5/500] | Time: 525.70s | LR: 1.0e-03 | Train Loss: 0.051145 | Val Loss: 0.044197 | Val NMSE: 16.37 dB
Epoch [6/500] | Time: 522.07s | LR: 1.0e-03 | Train Loss: 0.038934 | Val Loss: 0.032044 | Val NMSE: 14.96 dB
Epoch [7/500] | Time: 526.53s | LR: 1.0e-03 | Train Loss: 0.029931 | Val Loss: 0.026439 | Val NMSE: 14.13 dB
Epoch [8/500] | Time: 522.79s | LR: 1.0e-03 | Train Loss: 0.023441 | Val Loss: 0.020789 | Val NMSE: 13.08 dB
Epoch [9/500] | Time: 527.53s | LR: 1.0e-03 | Train Loss: 0.018666 

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def evaluate_snr_vs_nmse(snr_points, test_iterations=100):
    """
    Evaluates the model on the test set for specific SNR points.
    
    Args:
        snr_points (list): List of SNR values in dB to evaluate (e.g., [-10, -5, 0...])
        test_iterations (int): Number of batches to average over for each SNR point.
    """
    # Switch to evaluation mode
    atn.eval()
    encoder.eval()
    decoder.eval()
    stn.eval()
    
    nmse_results = []
    
    print(f"Starting evaluation on {len(snr_points)} SNR points...")
    
    with torch.no_grad():
        for snr_val in snr_points:
            total_nmse = 0.0
            
            for i in range(test_iterations):
                # 1. Fetch batch (Ignore the random SNR returned by collater)
                H_d, H_u, _ = collater(mode="test")
                
                # 2. Create fixed SNR tensor for this specific evaluation point
                # Shape must match [Batch, 1]
                current_batch_size = H_d.shape[0]
                fixed_snr = torch.full((current_batch_size, 1), snr_val, device=device).float()
                
                H_d = H_d.to(device)
                H_u = H_u.to(device)
                
                # 3. Forward Pass (Same as training but with fixed_snr)
                # ATN
                T = atn(H_d, fixed_snr)
                # Encoder
                c = encoder(T, fixed_snr)
                # Power Norm & R2C
                s = enc_to_complex_and_normalize(c)
                # Channel Simulation (using fixed_snr)
                s_hat = channel_sim(s, fixed_snr, H_u)
                # C2R
                c_hat = c2r(s_hat)
                # Decoder
                T_hat = decoder(c_hat, fixed_snr)
                # STN
                H_hat = stn(T_hat, fixed_snr)
                
                # 4. Calculate NMSE
                # Using the nmse_loss function defined in your training loop
                nmse = nmse_loss(H_d, H_hat)
                total_nmse += nmse.item()
            
            # Average NMSE for this SNR point
            avg_nmse = total_nmse / test_iterations
            nmse_results.append(avg_nmse)
            print(f"SNR: {snr_val} dB | NMSE: {avg_nmse:.4f} dB")
            
    return nmse_results

# Define SNR points to evaluate
snr_range = [-10, -5, 0, 5, 10]

# Run evaluation (using fewer iterations for quick plotting, increase for full test)
nmse_values = evaluate_snr_vs_nmse(snr_range, test_iterations=VAL_ITERATIONS)

Starting evaluation on 5 SNR points...


In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
plt.plot(snr_range, nmse_values, marker='o', linewidth=2, label='ADJSCC-CSInet+')

plt.title(f'NMSE vs. SNR (Compression Ratio = {encoder.M * 2 * 16 / encoder.total_elements if hasattr(encoder, \"M\") else 16})')
plt.xlabel('SNR (dB)')
plt.ylabel('NMSE (dB)')
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.legend()
plt.xticks(snr_range)
plt.show()

# Print raw data for table usage
print("\n--- Final Results ---")
print(f"SNR (dB): {snr_range}")
print(f"NMSE (dB): {[float(f'{x:.2f}') for x in nmse_values]}")

In [ ]:
import os

def save_checkpoint(path="adjscc_csinet_checkpoint.pth"):
    """
    Saves the model state dictionaries and optimizer state.
    """
    checkpoint = {
        'atn_state_dict': atn.state_dict(),
        'encoder_state_dict': encoder.state_dict(),
        'decoder_state_dict': decoder.state_dict(),
        'stn_state_dict': stn.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }
    torch.save(checkpoint, path)
    print(f"Model checkpoint saved to {path}")

# Save the trained model
save_checkpoint("adjscc_csinet_final.pth")

Model checkpoint saved to adjscc_csinet_final.pth
